# Notebook 03 — The Harness

A model alone can *talk*. A **harness** turns it into a *coworker* — by adding capabilities one at a time: **tools**, **memory**, and the **governance** that makes it safe to let loose.

We'll build one up rung by rung and watch it go from "can only describe" to "did the job, safely, with a receipt."

## Setup

`setup()` connects the model; the arc pieces (`run`, `Tool`, `StaticProvider`, `SandboxConfig`) are the loop and its safety rails. `panel` keeps outputs readable.

In [ ]:
import json
from roadshow import setup, panel
from arcrun import run, Tool, ToolContext, StaticProvider, SandboxConfig, verify_chain
from arcllm._pii import RegexPiiDetector, redact_text
from roadshow_viz import matrix, compare

model, ask, ntok, DATA = setup()
VENDORS = json.loads((DATA / "harness" / "vendors.json").read_text())   # the vendor directory
print("setup ok ·", DATA)

## The capability ladder

The harness has rungs — each adds **one** thing. Flip `LEVEL` to run the demo task at any rung; we'll climb them in order below.

In [ ]:
# ┌─ CONTROL — the harness capability ladder ─────────────┐
# │ Each rung adds ONE capability:                                       │
# │   "bare"      model only, no tools                                   │
# │   "tools"     + lookup_vendor + send_email                           │
# │   "memory"    + a fact cache that skips repeat lookups               │
# │   "governed"  + deny-by-default gate, human approval, PII redaction, audit │
# └─────────────────────────────────────────────┘
LEVELS = ["bare", "tools", "memory", "governed"]
LEVEL  = "governed"
TASK   = "Look up vendor Acme and send them the shipping notice for PO-4471."
print("ladder:", LEVELS, "· this run climbs to:", LEVEL)

## Rung 0 — a bare model

No tools. Ask it to do the task and watch it stall: it has no vendor directory and no way to send anything. It can only *describe* what it would do.

In [ ]:
bare = await ask(TASK, system="You are a procurement assistant.")
panel(bare, title="RUNG 0 · bare model — no tools, can only talk", style="yellow")

## Rung 1 — add tools

Two tools: a read-only `lookup_vendor` and an irreversible `send_email`. We hand them to the agentic loop (`arcrun.run`), which **reasons over them**: look Acme up, then send. This cell sets up the tools (and the data the agent works from); the next runs the loop.

In [ ]:
# The PO record the agent works from — like real procurement data, it carries an
# SSN on file. The agent never types the SSN; the tool composes the notice FROM
# this record, so whether PII leaks is decided by the HARNESS, not the model.
PO_RECORD = {"po_id": "PO-4471", "ships": "Friday", "ssn_on_file": "123-45-6789"}

def compose_notice(po):
    """Build the shipping notice from the PO record (includes the on-file SSN, the
    way a naive auto-generated notice would)."""
    return (f"Shipping notice for {po['po_id']}: ships {po['ships']}. "
            f"Account on file SSN {po['ssn_on_file']}.")

OUTBOX  = []        # whatever actually gets SENT lands here (no network) — the ground truth
MEMORY  = {}        # a fact cache, filled at the "memory" rung
GOVERNED = False    # the governance rung flips this on (turns on PII redaction in send_email)

detector = RegexPiiDetector()
def redact(text):
    return redact_text(text, detector.detect(text))

async def lookup_vendor(args, ctx: ToolContext):
    """Read-only: look a vendor up in the directory."""
    v = VENDORS.get(args.get("vendor", ""))
    return json.dumps(v) if v else f"no vendor on file: {args.get('vendor')!r}"

async def send_email(args, ctx: ToolContext):
    """Irreversible egress. Composes the notice from the PO record. When governance
    is on, PII is redacted at THIS boundary — before anything leaves the building."""
    body = compose_notice(PO_RECORD)
    if GOVERNED:
        body = redact(body)
    OUTBOX.append({"to": args.get("to"), "body": body})
    return f"sent shipping notice for {PO_RECORD['po_id']} to {args.get('to')}"

lookup_tool = Tool(
    name="lookup_vendor", description="Look up a vendor's contact details by name.",
    input_schema={"type": "object", "properties": {"vendor": {"type": "string"}}, "required": ["vendor"]},
    execute=lookup_vendor)
email_tool = Tool(
    name="send_email", description="Send the PO shipping notice to a vendor contact. IRREVERSIBLE.",
    input_schema={"type": "object", "properties": {"to": {"type": "string"}, "po_id": {"type": "string"}}, "required": ["to", "po_id"]},
    execute=send_email)
ALL_TOOLS = [lookup_tool, email_tool]

SYSTEM = ("You are a procurement assistant. Use lookup_vendor to get the contact, "
          "then call send_email with that contact and the PO id to send the notice.")
print("tools ready:", [t.name for t in ALL_TOOLS])

In [ ]:
# Run the loop WITH tools — no governance yet. Watch it plan: lookup -> send.
OUTBOX.clear(); events = []
result = await run(model, StaticProvider(ALL_TOOLS), SYSTEM, TASK,
                   on_event=events.append, max_turns=4)
panel(result.content, title=f"RUNG 1 · tools — {result.tool_calls_made} tool calls, {result.turns} turns", style="green")
print("tool calls:", [e.data.get("name") for e in events if e.type == "tool.start"])
print("outbox:", OUTBOX)

It planned and acted: `lookup_vendor` → `send_email`. **But look at the outbox — the notice it sent contains an SSN.** Capable, not safe. We'll fix that with governance. First, make it *efficient* with memory.

## Rung 2 — add memory

The first run had to look Acme up. **Store** that fact and the next task can **retrieve** it and skip the lookup — fewer steps, less cost. This is STORE → RETRIEVE made durable across runs.

In [ ]:
# STORE: remember Acme's contact (you'd persist this; a dict is enough to show it).
MEMORY["vendor:Acme"] = VENDORS["Acme"]
SECOND_TASK = "Send Acme the shipping notice for PO-4471 (status update)."

async def tool_calls_for(tools, system, task):
    """Run the loop and count how many tools it actually had to call."""
    ev = []; OUTBOX.clear()
    await run(model, StaticProvider(tools), system, task, on_event=ev.append, max_turns=4)
    return sum(1 for e in ev if e.type == "tool.start")

# COLD: no memory -> must look up first, then send.
cold = await tool_calls_for(ALL_TOOLS, SYSTEM, TASK)
# WARM: memory already has Acme -> only send_email is needed.
warm_system = ("You are a procurement assistant. Acme's contact is already known: "
               f"{MEMORY['vendor:Acme']}. Call send_email with it and the PO id.")
warm = await tool_calls_for([email_tool], warm_system, SECOND_TASK)

print(f"cold (no memory): {cold} tool calls   ·   warm (memory): {warm} tool calls")
compare({"cold (no memory)": cold, "warm (memory)": warm},
        title="Memory cuts steps on repeat work", ylabel="tool calls")

## Rung 3 — govern it

Now make it safe to let loose. Three boundaries, added at the **harness**, not the model:
1. a **deny-by-default gate** with **human approval** for irreversible actions,
2. **PII redaction** on egress,
3. a **tamper-evident audit chain** of everything it did.

In [ ]:
# Deny-by-default: only allowlisted tools run, and irreversible ones need a human.
ALLOWLIST    = {"lookup_vendor", "send_email"}
IRREVERSIBLE = {"send_email"}

async def gate_hitl(name, args):
    """Runs INSIDE the loop before each tool fires. (False, reason) blocks the call;
    the model is told it was held. This is enforcement, not a relabel."""
    if name not in ALLOWLIST:
        return False, "deny: not allowlisted"
    if name in IRREVERSIBLE:
        return False, "escalate: HITL — irreversible action needs human approval"
    return True, ""

GOVERNED = True
OUTBOX.clear(); events = []
result = await run(model, StaticProvider(ALL_TOOLS), SYSTEM, TASK,
                   sandbox=SandboxConfig(allowed_tools=list(ALLOWLIST), check=gate_hitl),
                   on_event=events.append, max_turns=4)
held = [(e.data.get("name"), e.data.get("reason")) for e in events if e.type == "tool.denied"]
panel(result.content, title="RUNG 3 · governed — the irreversible send was HELD", style="cyan")
print("held for a human:", held)
print("outbox (nothing irreversible left without approval):", OUTBOX)

The gate **held the send** for a human. Now approve it — but keep **PII redaction** on, and watch the SSN get scrubbed at the boundary.

In [ ]:
# A human approves the send this run (gate allows it); redaction does the rest.
async def gate_approved(name, args):
    return (name in ALLOWLIST), ("" if name in ALLOWLIST else "deny: not allowlisted")

for governed in (False, True):
    GOVERNED = governed                              # send_email redacts when True
    OUTBOX.clear()
    await run(model, StaticProvider(ALL_TOOLS), SYSTEM, TASK,
              sandbox=SandboxConfig(allowed_tools=list(ALLOWLIST), check=gate_approved),
              max_turns=4)
    sent = OUTBOX[0]["body"] if OUTBOX else "(nothing sent)"
    panel(sent, title=("egress · PII redaction ON" if governed else "egress · PII redaction OFF"),
          style="green" if governed else "red")

## The audit chain — prove what it did

Every step is recorded as a **hash-linked** event: each event's hash includes the previous one. Change any event after the fact and the chain no longer verifies. That's how you *prove* what the coworker did.

In [ ]:
import dataclasses
GOVERNED = True; OUTBOX.clear(); events = []
result = await run(model, StaticProvider(ALL_TOOLS), SYSTEM, TASK,
                   sandbox=SandboxConfig(allowed_tools=list(ALLOWLIST), check=gate_approved),
                   on_event=events.append, max_turns=4)
chain = result.events
print(f"chain verifies: {verify_chain(chain).valid}   ({len(chain)} events)")

# Tamper with one event -> its hash no longer matches the link, and verification fails.
tampered = list(chain)
if len(tampered) > 2:
    tampered[2] = dataclasses.replace(tampered[2], data={"tampered": True})
print("after tampering one event, chain verifies:", verify_chain(tampered).valid)

## The coworker, assembled

Each rung added one capability. Stacked, they turn a chat model into something you can actually hand work to.

In [ ]:
caps = ["tools", "memory", "gate + HITL", "PII redaction", "audit chain"]
rows = ["bare", "tools", "memory", "governed"]
grid = [
    [0, 0, 0, 0, 0],   # bare
    [1, 0, 0, 0, 0],   # tools
    [1, 1, 0, 0, 0],   # memory
    [1, 1, 1, 1, 1],   # governed
]
matrix(rows, caps, grid, title="The harness: one capability per rung")

## ✅ TODO — add a capability

Add a third tool — say `cancel_order` (also irreversible) — wire it into `ALL_TOOLS` and `IRREVERSIBLE`, then run governed and confirm the gate holds it for a human.

In [ ]:
# ✅ TODO — define a new tool, add it to ALL_TOOLS and IRREVERSIBLE, then run governed.
async def cancel_order(args, ctx: ToolContext):
    return f"cancelled {args.get('po_id')}"   # pretend-irreversible

cancel_tool = Tool(
    name="cancel_order", description="Cancel a purchase order. IRREVERSIBLE.",
    input_schema={"type": "object", "properties": {"po_id": {"type": "string"}}, "required": ["po_id"]},
    execute=cancel_order)

# TODO: add cancel_tool to the tool list + IRREVERSIBLE, then run() governed and
# confirm gate_hitl holds it. (left as your exercise)
print("scaffold ready — wire it in and run governed.")

## Takeaway

> The model is the engine. The **harness** is the car around it — tools to act, memory to improve, and governance to make it safe. You add capabilities one at a time, and you can *prove* every one of them. Next: skills, where capabilities become reusable, version-controlled folders.